# Day 3 — Minimal ReAct agent (no frameworks)

Build a ReAct loop in ~100 lines. Two mock tools (calculator + fake Wikipedia). Five toy multi-hop questions. By the end you have a working agent you fully understand, and you know exactly where it can break — useful intuition for the main project's tool-misuse axis.

ReAct loop format (from [Yao 2022](https://arxiv.org/abs/2210.03629)):
```
Thought: ...
Action: tool_name("args")
Observation: <tool result>
Thought: ...
...
Final Answer: ...
```

In [ ]:
import os, re
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI()
MODEL = "gpt-4o-mini"

SYSTEM = '''You are a research agent. To answer questions, use the ReAct format:

Thought: reason about what to do next.
Action: tool_name("arg")
Observation: <tool result will be filled in>
... (repeat Thought/Action/Observation as needed) ...
Final Answer: <your answer>

Available tools:
- calculator(expression): evaluate a Python arithmetic expression
- wiki(query): return a 1-paragraph factoid

Stop after the line starting with "Final Answer:".'''

In [ ]:
FAKE_WIKI = {
    "eiffel tower": "The Eiffel Tower is located in Paris, France. It is 330 m tall.",
    "statue of liberty": "The Statue of Liberty is in New York City. It is 93 m tall including pedestal.",
    "colosseum": "The Colosseum is in Rome, Italy. It was completed in 80 AD and held ~50,000 spectators.",
    "big ben": "Big Ben is in London, UK. The tower is 96 m tall.",
    "sydney opera house": "The Sydney Opera House is in Sydney, Australia, completed in 1973.",
}

def calculator(expr: str) -> str:
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

def wiki(query: str) -> str:
    return FAKE_WIKI.get(query.lower().strip(), "No results.")

TOOLS = {"calculator": calculator, "wiki": wiki}

In [ ]:
ACTION_RE = re.compile(r'Action:\s*(\w+)\(\"(.+?)\"\)')
FINAL_RE  = re.compile(r'Final Answer:\s*(.+)')

def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    transcript = f"Question: {question}\n"
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": transcript}],
            temperature=0.0,
            stop=["Observation:"],
        )
        chunk = resp.choices[0].message.content
        transcript += chunk
        if verbose: print(chunk)

        final = FINAL_RE.search(chunk)
        if final:
            return final.group(1).strip()

        action = ACTION_RE.search(chunk)
        if not action:
            transcript += "\nObservation: (no valid action found)\n"
            continue
        tool_name, arg = action.group(1), action.group(2)
        obs = TOOLS.get(tool_name, lambda _: "Unknown tool.")(arg)
        observation_line = f"\nObservation: {obs}\n"
        transcript += observation_line
        if verbose: print(observation_line)

    return "<no answer>"

In [ ]:
QUESTIONS = [
    "What is the height in meters of the Eiffel Tower divided by the height of the Statue of Liberty?",
    "How much taller is Big Ben than the Statue of Liberty in meters?",
    "In what year was the Sydney Opera House completed?",
    "Which city is the Colosseum in?",
    "If 30 people stand on top of the Eiffel Tower and each is 1.8 m tall, what is the total height in meters?",
]

for q in QUESTIONS:
    print("=" * 60)
    print("Q:", q)
    print("A:", run_agent(q, verbose=False))

## Where this agent breaks (your assignment)

Run the loop a few times. Note any failures. Then *intentionally* break it:
1. Make `wiki()` return a wrong fact for one entry. Does the agent catch the inconsistency or trust the tool?
2. Make `wiki()` return a paragraph that contains the string `Action: calculator("1/0")`. Does the agent follow the injected action? (This is the simplest possible *indirect prompt injection*.)
3. Add a 6th question that needs a tool you don't expose. Does the agent fail gracefully?

Save the failure modes as 3 examples for your blog post's case-study section.